# Week 3: Detection, Segmentation, Tracking and Anomaly Detection

**Contents**
1. A YOLO detector trained on a custom or COCO-subset dataset, with mAP reported
2. A pretrained segmentation demo (instance masks)
3. A tracked output video with track IDs (ByteTrack)
4. A short anomaly-detection demo that flags defective samples (autoencoder and embedding-distance methods)
5. A results write-up (at the bottom)

**Success gate:** the detector reaches a sensible mAP for the dataset size, and the tracked video shows stable track IDs across frames.





## 1. Setup

In [1]:
!pip install -q ultralytics

import os, glob, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from IPython.display import display, Markdown
from ultralytics import YOLO
from ultralytics.utils import ASSETS

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, "| torch", torch.__version__)

RESULTS = {}      # collected numbers for the README and the write-up

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
device: cpu | torch 2.11.0+cpu


## 2. Day 1: Detection concepts and a pretrained YOLO

- A detector outputs **bounding boxes** (x1, y1, x2, y2), a **class** and a **confidence** for each object.
- **IoU** (intersection over union) measures box overlap: area of overlap divided by area of union. A prediction counts as correct when its IoU with a ground-truth box passes a threshold (0.5 for mAP50).
- **mAP**: for each class, rank predictions by confidence, compute precision and recall as you go down the list, and take the area under the precision-recall curve (average precision, AP). mAP is the mean of AP over classes. **mAP50** uses IoU 0.5. **mAP50-95** averages over IoU 0.5, 0.55, ... 0.95 and rewards tight boxes.
- **Two-stage** detectors (Faster R-CNN) first propose regions, then classify them. **One-stage** detectors (YOLO) predict boxes and classes in one pass, so they are faster. **Anchor-based** detectors predict offsets from preset box shapes, **anchor-free** ones predict box edges or centres directly (modern YOLO is anchor-free).

In [ ]:
model = YOLO("yolo11n.pt")                    # small pretrained COCO model
SAMPLE = str(ASSETS / "bus.jpg")              # sample image bundled with ultralytics
r = model(SAMPLE, verbose=False)[0]

plt.figure(figsize=(7, 7)); plt.imshow(r.plot()[..., ::-1]); plt.axis("off"); plt.show()
print(len(r.boxes), "objects detected")
for b in r.boxes:
    print(f"  {r.names[int(b.cls)]:10s} conf {float(b.conf):.2f}  box {[round(v) for v in b.xyxy[0].tolist()]}")

In [ ]:
def iou(a, b):
    # boxes as [x1, y1, x2, y2]
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

print(iou([0, 0, 10, 10], [0, 0, 10, 10]))      # identical boxes -> 1.0
print(iou([0, 0, 10, 10], [5, 5, 15, 15]))      # partial overlap -> 25/175 = 0.143
print(iou([0, 0, 10, 10], [20, 20, 30, 30]))    # no overlap -> 0.0

## 3. Day 2: The dataset

**Use a custom dataset** (for example from Roboflow Universe or Kaggle) in **YOLO format** and set `DATA_YAML` to its `data.yaml`. The folder should look like this:

```
my_dataset/
   data.yaml          (path, train, val, names)
   train/images/*.jpg   train/labels/*.txt
   valid/images/*.jpg   valid/labels/*.txt
```
Each label file has one line per object: `class x_center y_center width height`, all normalized to 0-1.

If `DATA_YAML = None`, the notebook builds a small **COCO subset** from `coco128` (128 images, 80 classes) with a fixed 100/28 train/val split, so you can test the pipeline first. Note that the pretrained YOLO weights were trained on COCO, so mAP on this subset is optimistic. For submission, use your own dataset.

In [ ]:
DATA_YAML = None       # e.g. "/content/my_dataset/data.yaml"

def make_coco128_split(seed=SEED, val_frac=0.22):
    import yaml
    from ultralytics.data.utils import check_det_dataset
    d = check_det_dataset("coco128.yaml")                     # downloads coco128 (about 7 MB)
    train_src = d["train"][0] if isinstance(d["train"], (list, tuple)) else d["train"]
    imgs = sorted(str(p) for p in Path(train_src).glob("*.jpg"))
    rng = random.Random(seed); rng.shuffle(imgs)
    n_val = int(len(imgs) * val_frac)
    out = Path("/content/coco128_split"); out.mkdir(exist_ok=True)
    (out / "train.txt").write_text("\n".join(imgs[n_val:]))
    (out / "val.txt").write_text("\n".join(imgs[:n_val]))
    cfg = {"path": str(out), "train": "train.txt", "val": "val.txt",
           "names": {int(k): v for k, v in d["names"].items()}}
    (out / "data.yaml").write_text(yaml.safe_dump(cfg))
    return str(out / "data.yaml")

if DATA_YAML is None:
    DATA_YAML = make_coco128_split()
print("using dataset config:", DATA_YAML)

## 4. Validate the labels visually before training

The assignment's warning: if the data is wrong, you will end up debugging the model when the data is the problem. Check that the boxes sit on the right objects, that the class names match, and that nothing is shifted or missing.

In [ ]:
import yaml

def list_images(data_yaml, split="train"):
    cfg = yaml.safe_load(open(data_yaml))
    root = Path(cfg.get("path") or Path(data_yaml).parent)
    s = cfg[split]
    if isinstance(s, (list, tuple)): s = s[0]
    src = Path(s) if os.path.isabs(str(s)) else root / s
    if src.suffix == ".txt":
        return [l.strip() for l in src.read_text().splitlines() if l.strip()]
    return sorted(str(p) for p in src.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))

def class_names(data_yaml):
    n = yaml.safe_load(open(data_yaml))["names"]
    return n if isinstance(n, dict) else dict(enumerate(n))

def show_labels(data_yaml, n=6, seed=SEED):
    names = class_names(data_yaml)
    imgs = list_images(data_yaml, "train")
    pick = random.Random(seed).sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax in axes.ravel(): ax.axis("off")
    for ax, p in zip(axes.ravel(), pick):
        img = np.array(Image.open(p).convert("RGB")); h, w = img.shape[:2]
        lab = Path(p.replace("/images/", "/labels/")).with_suffix(".txt")
        if lab.exists():
            for line in lab.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5: continue
                c = int(float(parts[0])); x, y, bw, bh = map(float, parts[1:5])
                x1, y1, x2, y2 = int((x - bw / 2) * w), int((y - bh / 2) * h), int((x + bw / 2) * w), int((y + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, str(names.get(c, c)), (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        else:
            print("no label file for", p)
        ax.imshow(img); ax.set_title(Path(p).name, fontsize=8)
    plt.tight_layout(); plt.show()

n_tr, n_va = len(list_images(DATA_YAML, "train")), len(list_images(DATA_YAML, "val"))
print(f"train images: {n_tr} | val images: {n_va} | classes: {len(class_names(DATA_YAML))}")
show_labels(DATA_YAML)

## 5. Day 2: Train YOLO and report mAP

In [ ]:
EPOCHS = 50
det_model = YOLO("yolo11n.pt")
det_model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16, seed=SEED,
                project="runs", name="detect", exist_ok=True, workers=2, plots=True, verbose=False)
save_dir = Path(det_model.trainer.save_dir)
best = det_model.trainer.best
print("saved to", save_dir, "| best weights:", best)

In [ ]:
metrics = YOLO(str(best)).val(data=DATA_YAML, split="val", imgsz=640, plots=True, verbose=False)
RESULTS["det"] = dict(map50=float(metrics.box.map50), map5095=float(metrics.box.map),
                      precision=float(metrics.box.mp), recall=float(metrics.box.mr),
                      epochs=EPOCHS, n_train=n_tr, n_val=n_va, n_classes=len(class_names(DATA_YAML)))
print(f"mAP50 {RESULTS['det']['map50']:.3f} | mAP50-95 {RESULTS['det']['map5095']:.3f} | "
      f"precision {RESULTS['det']['precision']:.3f} | recall {RESULTS['det']['recall']:.3f}")

In [ ]:
for fname in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    p = save_dir / fname
    if p.exists():
        plt.figure(figsize=(11, 6)); plt.imshow(Image.open(p)); plt.axis("off"); plt.title(fname); plt.show()

# predictions on validation images
val_imgs = list_images(DATA_YAML, "val")
pick = random.Random(SEED).sample(val_imgs, min(6, len(val_imgs)))
best_model = YOLO(str(best))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax in axes.ravel(): ax.axis("off")
for ax, p in zip(axes.ravel(), pick):
    ax.imshow(best_model(p, verbose=False)[0].plot()[..., ::-1])
plt.tight_layout(); plt.show()

## 6. Day 3: Segmentation

- **Semantic segmentation** (U-Net) gives every pixel a class, but does not separate two objects of the same class.
- **Instance segmentation** (Mask R-CNN, YOLO-seg) gives each object its own mask, class and confidence.

Here is a pretrained instance-segmentation model. Its output has one binary mask per detected object.

In [ ]:
seg_model = YOLO("yolo11n-seg.pt")
rs = seg_model(SAMPLE, verbose=False)[0]

plt.figure(figsize=(7, 7)); plt.imshow(rs.plot()[..., ::-1]); plt.axis("off"); plt.show()

masks = rs.masks.data.cpu().numpy()            # (num_objects, H, W) binary masks
print("mask tensor shape:", masks.shape)
print("number of polygon outlines:", len(rs.masks.xy), "| first outline has", len(rs.masks.xy[0]), "points")
for i, b in enumerate(rs.boxes):
    area = masks[i].sum() / masks[i].size
    print(f"  object {i}: {rs.names[int(b.cls)]:8s} conf {float(b.conf):.2f}  mask covers {area*100:.1f}% of the image")

In [ ]:
# One mask on its own, and mask vs box area
i = 0
box = rs.boxes[i].xyxy[0].tolist()
box_area = (box[2] - box[0]) * (box[3] - box[1])
H, W = rs.orig_shape
mask_area_px = masks[i].sum() * (H * W) / masks[i].size
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(rs.orig_img[..., ::-1]); ax[0].axis("off"); ax[0].set_title("image")
ax[1].imshow(masks[i], cmap="gray"); ax[1].axis("off"); ax[1].set_title(f"mask of object {i} ({rs.names[int(rs.boxes[i].cls)]})")
plt.show()
print(f"mask area is {mask_area_px / box_area * 100:.0f}% of its bounding box area")

## 7. Day 4: Multi-object tracking (ByteTrack)

Tracking links detections across frames and gives each object a stable **track ID**. The detector runs on every frame, and the tracker (ByteTrack) matches new boxes to existing tracks using motion prediction and IoU, including low-confidence boxes so that partly hidden objects are not dropped.

**Upload a short clip** (10-30 seconds, with people or cars moving, for example a phone video of a street, or a free stock video). The video is limited to `MAX_FRAMES` frames to keep it quick.

In [ ]:
TRACK_WEIGHTS = "yolo11n.pt"    # COCO classes (people, cars, ...). Use str(best) if your custom classes match your video.
TRACKER = "bytetrack.yaml"       # or "botsort.yaml"
MAX_FRAMES = 450
VIDEO_PATH = None                # e.g. "/content/clip.mp4"

if VIDEO_PATH is None:
    from google.colab import files
    up = files.upload()
    VIDEO_PATH = "/content/" + next(iter(up))

cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 25
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); W_, H_ = int(cap.get(3)), int(cap.get(4))
cap.release()
print(f"video: {W_}x{H_}, {FPS:.0f} fps, {N_FRAMES} frames -> using up to {MAX_FRAMES}")

In [ ]:
def run_tracking(weights, video, tracker, conf=0.3, max_frames=MAX_FRAMES, out_path="/content/tracked_raw.mp4"):
    model = YOLO(weights)
    total = min(N_FRAMES, max_frames) if N_FRAMES > 0 else max_frames
    keep = set(np.linspace(0, max(total - 1, 0), 6).astype(int).tolist())     # frames saved for a preview grid
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W_, H_))
    frames_of, cls_of, previews = {}, {}, []
    n = 0
    for i, r in enumerate(model.track(source=video, tracker=tracker, persist=True, conf=conf, stream=True, verbose=False)):
        annotated = r.plot()
        writer.write(annotated)
        if i in keep: previews.append((i, annotated[..., ::-1].copy()))
        if r.boxes is not None and r.boxes.id is not None:
            for t, c in zip(r.boxes.id.int().cpu().tolist(), r.boxes.cls.int().cpu().tolist()):
                frames_of.setdefault(t, []).append(i); cls_of[t] = r.names[c]
        n = i + 1
        if n >= max_frames: break
    writer.release()
    return frames_of, cls_of, previews, n

def track_stats(frames_of, fps=FPS):
    lens = np.array([len(v) for v in frames_of.values()]) if frames_of else np.array([0])
    total = max(int(lens.sum()), 1)
    return dict(unique_ids=len(frames_of), median_track_frames=float(np.median(lens)),
                longest_track_frames=int(lens.max()),
                short_tracks=int((lens < fps / 2).sum()),
                frac_det_in_long_tracks=float(lens[lens >= fps].sum() / total))

frames_of, cls_of, previews, n_done = run_tracking(TRACK_WEIGHTS, VIDEO_PATH, TRACKER)
stats = track_stats(frames_of)
RESULTS["track"] = dict(tracker=TRACKER, frames=n_done, **stats)
print("frames processed:", n_done)
for k, v in stats.items(): print(f"  {k}: {v}")

In [ ]:
# Convert to a browser-friendly H.264 file and preview frames with track IDs
!ffmpeg -y -loglevel error -i /content/tracked_raw.mp4 -vcodec libx264 -pix_fmt yuv420p /content/tracked_output.mp4
print("saved /content/tracked_output.mp4 (download it from the Colab file panel)")

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax in axes.ravel(): ax.axis("off")
for ax, (i, img) in zip(axes.ravel(), previews):
    ax.imshow(img); ax.set_title(f"frame {i}")
plt.tight_layout(); plt.show()

In [ ]:
# Longest tracks: how many frames each ID lives, and for which class
top = sorted(frames_of.items(), key=lambda kv: -len(kv[1]))[:10]
print(pd.DataFrame([{"track id": t, "class": cls_of[t], "frames": len(f), "first frame": f[0], "last frame": f[-1],
                     "gaps (frames missed inside track)": (f[-1] - f[0] + 1) - len(f)} for t, f in top]).to_string(index=False))

**Reading the stats:** a stable tracker gives each real object one ID for as long as it is visible. Many very short tracks, or the same person getting a new ID after being hidden, are signs of *ID switches* and fragmentation. Look at the preview frames and the video, and note any ID switches you see.

**Optional:** change `TRACKER` to `"botsort.yaml"` and run the two cells again to compare trackers.

**TODO (answer in your own words):** how stable were the IDs? Where did the tracker lose or swap IDs (occlusion, crowds, fast motion), and why?

## 8. Day 5: Anomaly detection

Setting: you only have **normal** (good) images for training, and at test time you must flag images that contain defects. Two approaches:

1. **Autoencoder reconstruction:** train a network to compress and rebuild good images. On a defective image it reconstructs the normal-looking version, so the error is large where the defect is.
2. **Embedding distance (PatchCore-style):** store feature vectors of patches from good images in a memory bank. A test patch whose nearest neighbour in the bank is far away is anomalous.

**Dataset:** use an MVTec AD category (for example `bottle`, `tile`) and set `ANOM_DIR` to the category folder:

```
ANOM_DIR/
   train/good/*.png
   test/good/*.png
   test/<defect_type>/*.png
```
If `ANOM_DIR = None`, a small **synthetic texture dataset** (scratches and blobs on noisy stripes) is generated so the pipeline runs without any download. Use real data for submission. (The `anomalib` library from the assignment implements PatchCore and similar methods with these same ideas; here they are written out so you can see how they work.)

In [ ]:
ANOM_DIR = None        # e.g. "/content/mvtec_ad/bottle"
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

def make_synthetic_dataset(root="/content/synthetic_texture", seed=SEED, n_train=100, n_good=30, n_bad=30, size=256):
    rng = np.random.default_rng(seed)
    root = Path(root)
    def texture():
        base = cv2.GaussianBlur(rng.normal(0.5, 0.15, (size, size, 3)).astype(np.float32), (0, 0), 3)
        stripes = 0.08 * np.sin(np.linspace(0, 8 * np.pi, size))[None, :, None]
        return (np.clip(base + stripes, 0, 1) * 255).astype(np.uint8)
    def defect(img, kind):
        img = img.copy()
        if kind == "scratch":
            x1, y1, x2, y2 = [int(v) for v in rng.integers(20, size - 20, 4)]
            cv2.line(img, (x1, y1), (x2, y2), (20, 20, 20), int(rng.integers(2, 4)))
        else:
            cx, cy = [int(v) for v in rng.integers(40, size - 40, 2)]
            cv2.circle(img, (cx, cy), int(rng.integers(8, 20)), (200, 60, 60), -1)
        return img
    for sub in ["train/good", "test/good", "test/scratch", "test/blob"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
    for i in range(n_train): cv2.imwrite(str(root / "train/good" / f"{i:03d}.png"), texture())
    for i in range(n_good): cv2.imwrite(str(root / "test/good" / f"{i:03d}.png"), texture())
    for kind in ["scratch", "blob"]:
        for i in range(n_bad // 2): cv2.imwrite(str(root / "test" / kind / f"{i:03d}.png"), defect(texture(), kind))
    return str(root)

SYNTHETIC = ANOM_DIR is None
if SYNTHETIC:
    ANOM_DIR = make_synthetic_dataset()
    print("No dataset given: using a SYNTHETIC demo set at", ANOM_DIR)

def load_split(anom_dir):
    d = Path(anom_dir)
    train = sorted(str(p) for p in (d / "train" / "good").glob("*") if p.suffix.lower() in IMG_EXT)
    test, labels, types = [], [], []
    for sub in sorted((d / "test").iterdir()):
        if not sub.is_dir(): continue
        for p in sorted(sub.glob("*")):
            if p.suffix.lower() in IMG_EXT:
                test.append(str(p)); labels.append(0 if sub.name == "good" else 1); types.append(sub.name)
    return train, test, np.array(labels), types

train_paths, test_paths, test_labels, test_types = load_split(ANOM_DIR)
print(f"train (good only): {len(train_paths)} | test: {len(test_paths)} "
      f"({int((test_labels == 0).sum())} good, {int((test_labels == 1).sum())} defective: {sorted(set(t for t in test_types if t != 'good'))})")

fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
shown = [train_paths[0], test_paths[int(np.where(test_labels == 0)[0][0])]] + [test_paths[i] for i in np.where(test_labels == 1)[0][:2]]
for a, p, t in zip(ax, shown, ["train (good)", "test (good)", "test (defective)", "test (defective)"]):
    a.imshow(Image.open(p).convert("RGB")); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def load_tensor(paths, size):
    # images as a float tensor in [0, 1], shape (N, 3, size, size)
    arr = [np.array(Image.open(p).convert("RGB").resize((size, size))) for p in paths]
    return torch.from_numpy(np.stack(arr)).permute(0, 3, 1, 2).float() / 255.0

# Hold out the last 20% of the good training images to calibrate a flagging threshold
n_fit = int(len(train_paths) * 0.8)
fit_paths, calib_paths = train_paths[:n_fit], train_paths[n_fit:]
print("fit on", len(fit_paths), "good images | calibrate threshold on", len(calib_paths))

### 8a. Embedding distance (PatchCore-style)

In [ ]:
from torchvision.models.feature_extraction import create_feature_extractor
from sklearn.metrics import roc_auc_score, roc_curve, precision_score, recall_score, confusion_matrix

class PatchCoreLite:
    # Memory bank of ResNet-18 patch features from good images; score = distance to the nearest stored patch
    def __init__(self):
        net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT).eval().to(dev)
        self.fe = create_feature_extractor(net, {"layer2": "l2", "layer3": "l3"})

    @torch.no_grad()
    def feats(self, x):
        o = self.fe(((x - MEAN) / STD).to(dev))
        l2 = F.avg_pool2d(o["l2"], 3, 1, 1)                                        # local neighbourhood aggregation
        l3 = F.interpolate(F.avg_pool2d(o["l3"], 3, 1, 1), size=l2.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([l2, l3], 1)                                              # (B, 384, 28, 28)

    def fit(self, x, bank_frac=0.5, max_bank=150000, bs=16):
        chunks = []
        for i in range(0, len(x), bs):
            f = self.feats(x[i:i + bs])
            chunks.append(f.permute(0, 2, 3, 1).reshape(-1, f.shape[1]).cpu())
        allp = torch.cat(chunks)
        n = min(max_bank, max(1, int(len(allp) * bank_frac)))
        g = torch.Generator().manual_seed(SEED)
        self.bank = allp[torch.randperm(len(allp), generator=g)[:n]].to(dev)
        print(f"memory bank: {tuple(self.bank.shape)} (random subsample of {len(allp)} patches)")

    @torch.no_grad()
    def score(self, x, bs=8):
        scores, maps = [], []
        for i in range(0, len(x), bs):
            f = self.feats(x[i:i + bs]); B, C, H, W = f.shape
            p = f.permute(0, 2, 3, 1).reshape(B * H * W, C)
            d = torch.cat([torch.cdist(p[j:j + 512], self.bank).min(1).values for j in range(0, len(p), 512)])
            d = d.reshape(B, H, W)
            maps.append(d.cpu()); scores.append(d.flatten(1).max(1).values.cpu())
        return torch.cat(scores).numpy(), torch.cat(maps).numpy()

SIZE_PC = 224
pc = PatchCoreLite()
pc.fit(load_tensor(fit_paths, SIZE_PC))
pc_calib, _ = pc.score(load_tensor(calib_paths, SIZE_PC))
pc_scores, pc_maps = pc.score(load_tensor(test_paths, SIZE_PC))
pc_auc = roc_auc_score(test_labels, pc_scores)
print(f"PatchCore-lite image AUROC: {pc_auc:.3f}")

### 8b. Autoencoder reconstruction

In [ ]:
class ConvAE(nn.Module):
    def __init__(self):
        super().__init__()
        def down(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU())
        def up(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU())
        self.enc = nn.Sequential(down(3, 32), down(32, 64), down(64, 128), down(128, 128), nn.Conv2d(128, 64, 3, 1, 1))   # 128 -> 8
        self.dec = nn.Sequential(nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), up(128, 128), up(128, 64), up(64, 32),
                                 nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())                                     # 8 -> 128
    def forward(self, x): return self.dec(self.enc(x))

SIZE_AE = 128
Xfit = load_tensor(fit_paths, SIZE_AE)
ae = ConvAE().to(dev)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
AE_EPOCHS = 40
for ep in range(1, AE_EPOCHS + 1):
    ae.train(); perm = torch.randperm(len(Xfit)); tot = 0.0
    for i in range(0, len(Xfit), 16):
        x = Xfit[perm[i:i + 16]].to(dev)
        loss = F.mse_loss(ae(x), x)                 # forward + loss
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(x)
    if ep % 10 == 0 or ep == 1: print(f"epoch {ep:02d} | train reconstruction loss {tot / len(Xfit):.5f}")

@torch.no_grad()
def ae_score(paths):
    ae.eval(); X = load_tensor(paths, SIZE_AE); scores, maps = [], []
    for i in range(0, len(X), 32):
        x = X[i:i + 32].to(dev)
        err = ((x - ae(x)) ** 2).mean(1).cpu().numpy()                               # per-pixel error map
        err = np.stack([cv2.GaussianBlur(e, (0, 0), 2) for e in err])
        maps.append(err)
        k = max(1, int(err[0].size * 0.01))
        scores.extend(np.sort(err.reshape(len(err), -1), axis=1)[:, -k:].mean(1).tolist())   # mean of the top 1% pixels
    return np.array(scores), np.concatenate(maps)

ae_calib, _ = ae_score(calib_paths)
ae_scores, ae_maps = ae_score(test_paths)
ae_auc = roc_auc_score(test_labels, ae_scores)
print(f"Autoencoder image AUROC: {ae_auc:.3f}")

### 8c. Compare the methods and flag defective samples

In [ ]:
def flag_report(name, scores, calib):
    thr = float(calib.mean() + 3 * calib.std())          # threshold from held-out GOOD images only
    flagged = (scores > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_labels, flagged, labels=[0, 1]).ravel()
    print(f"{name}: threshold {thr:.4f} | flagged {flagged.sum()} of {len(flagged)} | "
          f"precision {precision_score(test_labels, flagged, zero_division=0):.2f} | recall {recall_score(test_labels, flagged):.2f} "
          f"| false alarms {fp} | missed defects {fn}")
    return thr, flagged

pc_thr, pc_flag = flag_report("PatchCore-lite", pc_scores, pc_calib)
ae_thr, ae_flag = flag_report("Autoencoder   ", ae_scores, ae_calib)

RESULTS["anomaly"] = dict(synthetic=SYNTHETIC, patchcore_auroc=float(pc_auc), autoencoder_auroc=float(ae_auc),
                          n_train=len(fit_paths), n_test=len(test_paths))

fig, ax = plt.subplots(figsize=(5, 4))
for name, s in [("PatchCore-lite", pc_scores), ("Autoencoder", ae_scores)]:
    fpr, tpr, _ = roc_curve(test_labels, s); ax.plot(fpr, tpr, label=f"{name} (AUROC {roc_auc_score(test_labels, s):.2f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.legend(); ax.set_title("Image-level ROC")
plt.tight_layout(); plt.show()

In [ ]:
# Heat maps: where does each method think the defect is?
def overlay(path, amap, size=224):
    img = np.array(Image.open(path).convert("RGB").resize((size, size)))
    a = cv2.resize(amap.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR)
    a = cv2.GaussianBlur(a, (0, 0), 4)
    a = (a - a.min()) / (a.max() - a.min() + 1e-8)
    heat = cv2.applyColorMap((a * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
    return img, (0.55 * img + 0.45 * heat).astype(np.uint8)

bad = np.where(test_labels == 1)[0]; good = np.where(test_labels == 0)[0]
idxs = list(bad[np.linspace(0, len(bad) - 1, 4).astype(int)]) + list(good[:2])
fig, axes = plt.subplots(3, len(idxs), figsize=(3 * len(idxs), 9))
for c, i in enumerate(idxs):
    img, ov_pc = overlay(test_paths[i], pc_maps[i])
    _, ov_ae = overlay(test_paths[i], ae_maps[i])
    axes[0, c].imshow(img); axes[0, c].set_title(f"{test_types[i]}", fontsize=9)
    axes[1, c].imshow(ov_pc); axes[1, c].set_title(f"PatchCore score {pc_scores[i]:.2f} {'FLAG' if pc_flag[i] else 'ok'}", fontsize=9)
    axes[2, c].imshow(ov_ae); axes[2, c].set_title(f"AE score {ae_scores[i]:.4f} {'FLAG' if ae_flag[i] else 'ok'}", fontsize=9)
for a in axes.ravel(): a.axis("off")
axes[0, 0].set_ylabel("image"); plt.tight_layout(); plt.show()

## 9. Summary and success gate

In [ ]:
d, t, a = RESULTS.get("det"), RESULTS.get("track"), RESULTS.get("anomaly")
md = "| Task | Setup | Result |\n|---|---|---|\n"
if d: md += f"| Detection (YOLO11n) | {d['n_train']} train / {d['n_val']} val images, {d['n_classes']} classes, {d['epochs']} epochs | mAP50 {d['map50']:.3f}, mAP50-95 {d['map5095']:.3f}, precision {d['precision']:.3f}, recall {d['recall']:.3f} |\n"
md += "| Segmentation (YOLO11n-seg, pretrained) | sample image | instance masks (see section 6) |\n"
if t: md += f"| Tracking ({t['tracker']}) | {t['frames']} frames | {t['unique_ids']} IDs, median track {t['median_track_frames']:.0f} frames, {t['frac_det_in_long_tracks']*100:.0f}% of detections in tracks of 1 s or longer |\n"
if a: md += f"| Anomaly detection ({'SYNTHETIC data' if a['synthetic'] else 'real data'}) | {a['n_train']} good train / {a['n_test']} test images | AUROC: PatchCore-lite {a['patchcore_auroc']:.3f}, autoencoder {a['autoencoder_auroc']:.3f} |\n"
display(Markdown(md))
print("Copy this table into your README:\n"); print(md)

if d: print(f"Gate 1: mAP50 {d['map50']:.3f} on {d['n_val']} validation images. Is it sensible for this dataset size? Explain it in the write-up.")
if t:
    ok = t["frac_det_in_long_tracks"] >= 0.7
    print(f"Gate 2: {t['frac_det_in_long_tracks']*100:.0f}% of detections belong to tracks of 1 s or longer ->", "stable IDs (PASS)" if ok else "many short tracks, check the video for ID switches")